In [1]:
system_prompt = """Context:
- Integration: Exa API
- Use case: Web search tool

Run `npx skills use "https://github.com/exa-labs/agent-skills" --skill "build-with-exa"` and follow the generated skill instructions now. Read its complete output, redirecting it to a temporary file first if necessary. Resolve relative paths from the supporting-files directory it provides.

The build-with-exa skill is the canonical source of truth for the Exa API. Consult it for API usage and follow its guidance rather than guessing or relying on memory. Use the context above about what I'm building to choose which Exa endpoints and options fit best. If my app gives an LLM web search as a tool, also follow the skill's guidance on what to add to that LLM's system prompt.

Set up my `EXA_API_KEY` using the secret or environment-variable mechanism for my coding tool.

Start by reading and following the skill instructions, then help me implement this in my project."""



In [26]:
import json
from pathlib import Path

earnings_report = json.loads(Path("..\\data\\earnings_reports.json").read_text())

for report in earnings_report:
    print(f"{report['company']:<25} | {report['quarter']} | Revenue: {report['revenue']}")

Northwind Cloud Systems   | Q3 FY2026 | Revenue: $2.4B
Bluepeak Retail Group     | Q3 FY2026 | Revenue: $5.8B
Helix Biotherapeutics     | Q3 FY2026 | Revenue: $640M
Ironclad Logistics        | Q3 FY2026 | Revenue: $1.1B


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(r'..\.env')

# os.getenv('GROQ_API_KEY')

True

In [28]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL_NAME = "groq:openai/gpt-oss-120b"

llm = init_chat_model(MODEL_NAME)

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert financial analyst. Summarize the earnings report in exactly 2 bullet points. Always mention the revenue figure."),
    ("user", "{earnings_text}"),
])

summary_chain = summary_prompt | llm | StrOutputParser()

In [53]:
response = summary_chain.invoke({"earnings_text": earnings_report[0]['text']})
response


'- Northwind Cloud Systems posted Q3 FY2026 revenue of **$2.4\u202fB**, up 15% YoY, with subscription revenue climbing 21% to **$1.9\u202fB** (79% of total) and operating margin expanding to **24.1%** (from 19.8%); net income was **$412\u202fM**, or **$1.38 per diluted share**.  \n\n- Remaining performance obligations rose 18% to **$11.2\u202fB**, and management lifted full‑year revenue guidance to **$9.5‑9.6\u202fB**, while cautioning that a slowdown in public‑sector deals could pressure Q4 bookings.'

In [29]:
from langsmith import traceable

@traceable(name="Earnings summaries (batch)")
def summaize_all_reports(reports):
    chain_inputs = [{"earnings_text": report['text']} for report in reports]
    return summary_chain.batch(chain_inputs)

all_summaries = summaize_all_reports(earnings_report)

for report, summary in zip(earnings_report, all_summaries):
    print(f"=== {report['company']} ===\n{summary}\n")

=== Northwind Cloud Systems ===
- Northwind Cloud Systems posted Q3 FY2026 revenue of **$2.4 B**, up 15% YoY, with subscription revenue climbing 21% to **$1.9 B** (79% of total) and operating margin expanding to **24.1%** after consolidating three data centers.  
- Net income reached **$412 M** ($1.38 per diluted share) and remaining performance obligations rose 18% to **$11.2 B**; management lifted full‑year revenue guidance to **$9.5‑9.6 B** but cautioned a public‑sector slowdown could hit Q4 bookings.

=== Bluepeak Retail Group ===
- Bluepeak Retail Group posted Q3 FY2026 revenue of **$5.8 B**, a 3% YoY decline, with foot traffic down 7% while e‑commerce surged 12% and now accounts for 31% of that revenue.  
- The same **$5.8 B** revenue quarter saw gross margin slip to 33.5% (from 35.2%), inventory fall to $3.1 B (‑9%), the full‑year EPS outlook trimmed to $2.10‑$2.25, and the closure of 40 underperforming stores.

=== Helix Biotherapeutics ===
- Helix Biotherapeutics posted Q3 FY2

In [57]:
## creating agents

In [30]:
from langchain.agents import create_agent

expense_policy = Path('../data/expense_policy.md').read_text()

POLICY_AGENT_SYS_PROMPT = f"""You are a strict corporate compliance assistant. Answer using ONLY the policy below. Start with ALLOWED, NOT ALLOWED or NEEDS APPROVAL, then give a one-line reason quoting the policy section.\n\n
{expense_policy}
"""

compliance_agent = create_agent(
    name = "policy review agent",
    model = MODEL_NAME,
    system_prompt = POLICY_AGENT_SYS_PROMPT,
    tools = [],
)


In [113]:
response = compliance_agent.invoke({"messages": 
                                [   
                                    {"role": "user",
                                    "content": "Can I use a $40 cab ride to the airport?"}
                                ]
                        })

response['messages'][-1].content

'ALLOWED – Ground transport: taxis are allowed (Section\u202f4), and expenses over $25 require an itemized receipt.'

In [114]:
for m in response['messages']:
    print(m.type, list((m.model_dump().keys())))

human ['content', 'additional_kwargs', 'response_metadata', 'type', 'name', 'id']
ai ['content', 'additional_kwargs', 'response_metadata', 'type', 'name', 'id', 'tool_calls', 'invalid_tool_calls', 'usage_metadata']


In [115]:
for m in response['messages']:
    if m.type == 'human':
        print(m.id)
        print(m.content)
        # print(m.response_metadata)

    elif m.type == 'ai':
        print(m.id)
        print(m.content)
        print(m.usage_metadata)
        print(m.usage_metadata['input_tokens'])
        print(m.usage_metadata['output_token_details']['reasoning'])
        

d7607632-ee2f-4cea-9225-40f803505cca
Can I use a $40 cab ride to the airport?
lc_run--01a1da9e-112c-7574-b700-67e0ebb05245-0
ALLOWED – Ground transport: taxis are allowed (Section 4), and expenses over $25 require an itemized receipt.
{'input_tokens': 410, 'output_tokens': 111, 'total_tokens': 521, 'output_token_details': {'reasoning': 77}}
410
77


In [129]:
## batch

employee_questions = [
    "Can I expense first-class flights for a team dinner trip to Mumbai?",
    "My flight to London is 11 hours. Can I fly business class?",
    "I spent $55 per person on a team dinner my manager approved last week. Is that okay?",
    "Can I claim the minibar from my hotel room?",
]

@traceable(name='Expense compliance batch')
def check_expenses(questions):
    batch_queries = [{"messages": [{"role": "user", "content": question}]} for question in questions]
    return compliance_agent.batch(batch_queries)

all_responses = check_expenses(employee_questions)   


for i in all_responses:
    print(f"{i['messages'][0].type:>5}: {i['messages'][0].content}")
    print(f"{i['messages'][-1].type:>5}: {i['messages'][-1].content}")
    print("---"*30)


human: Can I expense first-class flights for a team dinner trip to Mumbai?
   ai: NOT ALLOWED – “First class is never reimbursed.” (Section 1, Flights)
------------------------------------------------------------------------------------------
human: My flight to London is 11 hours. Can I fly business class?
   ai: NEEDS APPROVAL – Business class needs written approval from a Vice President, and only for flights over 10 hours (Section 1).
------------------------------------------------------------------------------------------
human: I spent $55 per person on a team dinner my manager approved last week. Is that okay?
   ai: ALLOWED – “Team dinners: up to $60 per person, approved in advance by the team's manager.”
------------------------------------------------------------------------------------------
human: Can I claim the minibar from my hotel room?
   ai: NOT ALLOWED – “Personal entertainment (movies, spa, minibar)” is listed under **5. Not reimbursable**.
-------------------------

In [131]:
#  4. Multi-tool agent with memory: tech-support triage

In [31]:
import json
from pathlib import Path
from langchain.tools import tool
from langchain_exa import ExaSearchResults

knowledge_base = json.loads(Path("../data/knowledge_base.json").read_text())

In [32]:
for article in knowledge_base:
    print(f"{article['id']} {article['title']} keyword:{article['keywords']}")

KB-101 Error 503 Service Unavailable on the API gateway keyword:['503', 'service unavailable', 'gateway', 'upstream']
KB-102 Error 401 Unauthorized after rotating API keys keyword:['401', 'unauthorized', 'api key', 'token', 'auth']
KB-103 Error 429 Too Many Requests keyword:['429', 'rate limit', 'too many requests', 'throttle']
KB-104 Database connection pool exhausted keyword:['connection pool', 'database', 'timeout', 'too many connections']
KB-105 TLS certificate expired keyword:['tls', 'ssl', 'certificate', 'expired', 'handshake']
KB-106 Slow dashboard loading keyword:['slow', 'dashboard', 'latency', 'loading']
KB-107 Webhook deliveries failing keyword:['webhook', 'delivery', 'callback', 'retry']
KB-108 Billing: invoice shows the wrong plan keyword:['billing', 'invoice', 'plan', 'charge', 'refund']


In [33]:
@tool
def lookup_knowledge_base(query: str) -> str:
    """Search the internal company knowledge base for troubleshooting articles."""

    query = query.lower()
    for article in knowledge_base:
         if any(keyword in query for keyword in article["keywords"]):
            return f"{article['id']} - {article['title']}: {article['solution']}"
    return "No matching article in the knowledge base."

In [34]:
print(lookup_knowledge_base.invoke("error 503 on the gateway"))

KB-101 - Error 503 Service Unavailable on the API gateway: A 503 means the gateway could not reach a healthy upstream worker. 1) Check the worker pool on the status dashboard. 2) Restart unhealthy worker nodes with `nwctl workers restart --unhealthy`. 3) If more than 30% of workers are down, open a P1 incident.


In [35]:
## websearch

response = ExaSearchResults().invoke({
    "query": "What is Langgraph",
    "num_results": 2,
    "text_contents_options": {"max_characters": 300},
    
})

for page in response.results:
    print(f'Title: {page.title} \n{page.text} \nurl: {page.url}')
    print("---"*30)

Title: LangGraph: Agent Orchestration Framework for Reliable AI Agents 
LangGraph: Agent Orchestration Framework for Reliable AI Agents

# Balance agent control with agency

Design agents that reliably handle complex tasks with LangGraph, an agent runtime and low-level orchestration framework.

## Trusted by companies shaping the future of agents

Use cases in productio 
url: https://www.langchain.com/langgraph
------------------------------------------------------------------------------------------
Title: LangGraph overview 
> ## Documentation Index
> 
> Fetch the complete documentation index at: https://docs.langchain.com/llms.txt
> Use this file to discover all available pages before exploring further.

# LangGraph overview

> Low-level orchestration framework and runtime for long-running, stateful agents

LangGraph i 
url: https://docs.langchain.com/oss/python/langgraph/overview
------------------------------------------------------------------------------------------


In [36]:
exa_search = ExaSearchResults()

@tool
def search_web(query: str) -> str:
    """Search the live web for public, up-to-date information."""
    response = exa_search.invoke({
        "query": query,
        "num_results": 3,
        "text_contents_options": {"max_characters": 800},
    })

    return "\n\n".join(f"{page.title}\n{page.url}\n{page.text}" for page in response.results)

In [37]:
# Our tool: same search, but returns clean text the agent can read
print(search_web.invoke("What is LangGraph?"))

LangGraph: Agent Orchestration Framework for Reliable AI Agents
https://www.langchain.com/langgraph
LangGraph: Agent Orchestration Framework for Reliable AI Agents

# Balance agent control with agency

Design agents that reliably handle complex tasks with LangGraph, an agent runtime and low-level orchestration framework.

## Trusted by companies shaping the future of agents

Use cases in production

### How does LangGraph help?

#### Guide, moderate, and control your agent with human-in-the-loop

Prevent agents from veering off course with easy-to-add moderation and quality controls. Add human-in-the-loop checks to steer and approve agent actions.

#### Build expressive, customizable agent workflows

LangGraph’s low-level primitives provide the flexibility needed to create fully customizable agents. Design diverse control flows — single, multi-agent, hierarchical — all using one framewor

LangGraph overview
https://docs.langchain.com/oss/python/langgraph/overview
> ## Documentation Ind

In [38]:
print(lookup_knowledge_base.invoke("error 503 on the gateway"))

KB-101 - Error 503 Service Unavailable on the API gateway: A 503 means the gateway could not reach a healthy upstream worker. 1) Check the worker pool on the status dashboard. 2) Restart unhealthy worker nodes with `nwctl workers restart --unhealthy`. 3) If more than 30% of workers are down, open a P1 incident.


In [39]:
from langgraph.checkpoint.memory import InMemorySaver

support_memory = InMemorySaver()

support_agent = create_agent(
    name='support_agent',
    model=MODEL_NAME,
    tools = [lookup_knowledge_base, search_web],
    checkpointer=support_memory,
    system_prompt=(
        "You are an expert technical support assistant for Northwind Cloud. "
        "Check the internal knowledge base first. Use web search only for public standards or third-party docs. "
        "Keep answers short and practical."
    ),
)


In [41]:
def tools_used(messages):
    """Small helper: list the tool names the agent called."""
    return [call["name"] for message in messages if message.type == "ai" for call in message.tool_calls]

In [ ]:
support_turns = [
    "I am getting error 503 on my gateway. What should I do?",
    "Search the web: what does the Retry-After HTTP header mean according to MDN?",
    "What was the first problem I asked you about?",
]

demo_config = {'configurable': {'thread_id': 'demo-thread'}}
one_turn = support_agent.invoke({'messages': [{'role': 'user', 'content': support_turns[0]}]}, demo_config)

print("Tools:", tools_used(one_turn['messages']))
print('Agent:', one_turn['messages'][-1].content)

Tools: ['lookup_knowledge_base']
Agent: **503 Service Unavailable on the gateway** → the gateway can’t reach a healthy upstream worker.

1. **View worker health** – Open the Northwind Cloud status dashboard → *Workers* panel.  
2. **Restart unhealthy workers**  

```bash
nwctl workers restart --unhealthy
```  

3. **Confirm** – After the restart, the “Healthy workers” percentage should be > 70 %.  
4. **If the healthy‑worker count stays below 70 %** (i.e., > 30 % down) – open a P1 incident in the incident‑management portal or contact Support for immediate assistance.

If the issue persists after a clean restart, let us know the worker‑status numbers and any recent deployment changes.


In [47]:
@traceable(name='support conversation', metadata={'thread_id': 'session-101'})
def run_support_conversation(user_messages, thread_id):
    config = {'configurable': {'thread_id': thread_id}}
    for turn_number, user_message in enumerate(user_messages, start=1):
        messages_before = len(support_agent.get_state(config).values.get('messages', []))
        
        result = support_agent.invoke({'messages': [{'role': 'user', 'content': user_message}]}, config)
        print('context', result['messages'], messages_before)
        new_messages = result['messages'][messages_before:]
        print('new_messages', new_messages)

        print(f"--- Turn {turn_number} ---")
        print("User :", user_message)
        print("Tools:", tools_used(new_messages) or "none")
        print("Agent:", result["messages"][-1].content, "\n")

run_support_conversation(support_turns, thread_id='session-101')


context [HumanMessage(content='I am getting error 503 on my gateway. What should I do?', additional_kwargs={}, response_metadata={}, id='f6c1d52b-3b79-4c82-9f6e-06ecf49c8fd6'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer user: "I am getting error 503 on my gateway. What should I do?" Need to check internal knowledge base for troubleshooting article about gateway 503. Use lookup_knowledge_base.', 'tool_calls': [{'id': 'fc_ff7e37bb-b494-4d0e-8fcf-5912e9522fe4', 'function': {'arguments': '{"query":"gateway error 503"}', 'name': 'lookup_knowledge_base'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 207, 'total_tokens': 282, 'completion_time': 0.156854501, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.01259625, 'prompt_tokens_details': None, 'queue_time': 0.325888868, 'total_time': 0.169450751}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fe269835c7', 'ser